# SemEval-2027 Task 12: ArtEmotion-28 — Starter Kit

A minimal starting point for the three subtasks. For each one: load the data,
train a very simple baseline, check it on the `val` split, and write a
submission file in the format Codabench expects. Replace the baselines with
your own systems.

| Subtask | Predict | Submission |
|---|---|---|
| **ST1** Cross-Cultural Emotion Prediction | one of eight emotions for a caption + painting | `predictions_text.csv` and `predictions_multimodal.csv` |
| **ST2** Cross-Cultural Sentiment Divergence | a language's share of positive annotators, given another language's | `predictions.csv` |
| **ST3** Image-Only Cross-Cultural Sentiment | a language's share of positive annotators, from the image alone | `predictions.csv` |

Each subtask section only needs the **Setup** section above it. Full task
description: [github.com/artemotion-28/artemotion-28](https://github.com/artemotion-28/artemotion-28)

## Setup

Run from the `starter_kit/` folder of the repository. Needs `pandas`,
`scikit-learn` and `pillow`, plus `torch` and `transformers` for the ST1
multimodal run.

In [1]:
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

REPO = Path("..")
EMOTIONS = ["amusement", "anger", "awe", "contentment", "disgust", "excitement", "fear", "sadness"]
POSITIVE = {"amusement", "awe", "contentment", "excitement"}

data = pd.read_csv(REPO / "annotations" / "artelingo28_train_val.csv")
data["language"] = data["language"].replace({"arabic": "Arabic", "chinese": "Chinese", "english": "English"})
data = data[data["emotion"].isin(EMOTIONS)]  # drop "something else" / "other"
data["positive"] = data["emotion"].isin(POSITIVE).astype(int)

train, val = data[data["split"] == "train"], data[data["split"] == "val"]
print(f"train: {len(train):,} captions, val: {len(val):,} captions, {data['language'].nunique()} languages")
data.head(3)

train: 152,447 captions, val: 8,057 captions, 28 languages


,image_id,genre,emotion,caption,art_style,painting,language,image_name,split,positive
0,0.0,abstract_painting,anger,"""အနီရောင်မှာ သွေးရောင်နှင့်တူနေသောကြောင့် "" 36",Color_Field_Painting,mark-rothko_no-17-1961(1),Burmese,Color_Field_Painting/mark-rothko_no-17-1961(1)...,train,0
5,0.0,abstract_painting,awe,dua warna berbeza namun tidak sama besar,Color_Field_Painting,mark-rothko_no-17-1961(1),Malay,Color_Field_Painting/mark-rothko_no-17-1961(1)...,train,1
10,0.0,abstract_painting,contentment,따뜻하고 아름다운 색채가 주는 느낌이 안정감 있고 따뜻하게 느껴진다.,Color_Field_Painting,mark-rothko_no-17-1961(1),Korean,Color_Field_Painting/mark-rothko_no-17-1961(1)...,train,1


---
## ST1 — Cross-Cultural Emotion Prediction

Predict the emotion of a caption. You submit two runs: a **text-only** run and
a **multimodal** run (the ranked one), which must use the image. Scored by
macro-F1 within each language, averaged over languages.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
from sklearn.svm import LinearSVC

def st1_score(frame, predictions):
    """Macro-F1 per language, averaged over languages."""
    frame = frame.assign(prediction=predictions)
    return np.mean([f1_score(g["emotion"], g["prediction"], average="macro")
                    for _, g in frame.groupby("language")])

# Text-only: character n-grams work for every script without a tokenizer.
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=100_000)
text_model = LinearSVC(class_weight="balanced")
text_model.fit(vectorizer.fit_transform(train["caption"]), train["emotion"])
print("text-only val macro-F1:", round(st1_score(val, text_model.predict(vectorizer.transform(val["caption"]))), 3))

text-only val macro-F1: 0.531


For the multimodal run, embed each painting once with a frozen pretrained
image model ([CLIP](https://huggingface.co/openai/clip-vit-base-patch32)) and
append the embedding to the text features. Embedding the 1,658 paintings takes
a few minutes on a CPU. Simpler image features, such as average colour, barely
change the predictions and would fail the image check below.

In [3]:
import torch
from scipy.sparse import csr_matrix, hstack
from transformers import CLIPModel, CLIPProcessor

clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").eval()
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

image_cache = {}
names = list(data["image_name"].unique())
for start in range(0, len(names), 32):
    batch = names[start:start + 32]
    with torch.no_grad():
        features = clip.get_image_features(**processor(
            images=[Image.open(REPO / "images" / n).convert("RGB") for n in batch], return_tensors="pt"))
    image_cache.update(zip(batch, torch.nn.functional.normalize(features, dim=-1).numpy()))

def multimodal_inputs(captions, images):
    return hstack([vectorizer.transform(captions), csr_matrix([2 * image_cache[i] for i in images])]).tocsr()

multimodal_model = LinearSVC(class_weight="balanced")
multimodal_model.fit(multimodal_inputs(train["caption"], train["image_name"]), train["emotion"])
print("multimodal val macro-F1:", round(st1_score(val, multimodal_model.predict(
    multimodal_inputs(val["caption"], val["image_name"]))), 3))

multimodal val macro-F1: 0.529


**Image check.** The test file contains hidden decoy rows: real captions paired
with another painting's image. A multimodal run that does not score lower on
them is excluded. Check yours the same way on `val`:

In [4]:
wrong_images = val["image_name"].sample(frac=1, random_state=0).values
shuffled = st1_score(val, multimodal_model.predict(multimodal_inputs(val["caption"], wrong_images)))
real = st1_score(val, multimodal_model.predict(multimodal_inputs(val["caption"], val["image_name"])))
print(f"image reliance: {real - shuffled:+.3f}  (must be above 0)")

image reliance: +0.026  (must be above 0)


**Submission.** Predict for the phase's input file in `phase_inputs/`
(columns `id,language,image,caption`; half the rows are hidden decoys, so
predict every row normally). For the Evaluation phase, change `dev` to `test`.

In [5]:
test = pd.read_csv(REPO / "phase_inputs" / "dev" / "st1_test.csv")
folder = Path("submission_st1")
folder.mkdir(exist_ok=True)
pd.DataFrame({"id": test["id"], "label": text_model.predict(vectorizer.transform(test["caption"]))}
             ).to_csv(folder / "predictions_text.csv", index=False)
pd.DataFrame({"id": test["id"], "label": multimodal_model.predict(multimodal_inputs(test["caption"], test["image"]))}
             ).to_csv(folder / "predictions_multimodal.csv", index=False)
shutil.make_archive("submission_st1", "zip", folder)
print("Upload submission_st1.zip to the ST1 competition.")

Upload submission_st1.zip to the ST1 competition.


---
## ST2 — Cross-Cultural Sentiment Divergence Prediction

For a painting, you get a **source** language's share of positive annotators;
predict the **target** language's share. Only (painting, language) pairs with
at least 5 annotators count. Scored by log loss (lower is better).

In [6]:
def shares(frame):
    """Share of positive annotators per (painting, language), with at least 5 annotators."""
    g = frame.groupby(["painting", "language"])["positive"].agg(k="sum", n="size").reset_index()
    g = g[g["n"] >= 5]
    return g.assign(p=g["k"] / g["n"])

def language_pairs(frame):
    s = shares(frame)
    pairs = s.merge(s, on="painting", suffixes=("_source", "_target"))
    return pairs[pairs["language_source"] != pairs["language_target"]]

def log_loss(q, k, n):
    q = np.clip(q, 0.01, 0.99)
    return np.mean(-(k * np.log2(q) + (n - k) * np.log2(1 - q)) / n)

train_pairs, val_pairs = language_pairs(train), language_pairs(val)
print(f"{len(train_pairs):,} training pairs, {len(val_pairs):,} val pairs")

161,668 training pairs, 8,472 val pairs


Baseline: average the source share with the target language's overall share
on the training paintings.

In [7]:
target_prior = shares(train).groupby("language")["p"].mean()

def st2_predict(source_share, target_language):
    return 0.5 * source_share + 0.5 * target_language.map(target_prior).fillna(0.7)

q = st2_predict(val_pairs["p_source"], val_pairs["language_target"])
print("val log loss:", round(log_loss(q, val_pairs["k_target"], val_pairs["n_target"]), 3),
      "| copying the source:", round(log_loss(val_pairs["p_source"], val_pairs["k_target"], val_pairs["n_target"]), 3))

val log loss: 0.793 | copying the source: 1.021


**Submission.** The input file has columns
`painting,source_language,target_language,source_p_positive`.

In [8]:
test = pd.read_csv(REPO / "phase_inputs" / "dev" / "st2_test.csv")
test["p_positive"] = st2_predict(test["source_p_positive"], test["target_language"])
folder = Path("submission_st2")
folder.mkdir(exist_ok=True)
test[["painting", "source_language", "target_language", "p_positive"]].to_csv(folder / "predictions.csv", index=False)
shutil.make_archive("submission_st2", "zip", folder)
print("Upload submission_st2.zip to the ST2 competition.")

Upload submission_st2.zip to the ST2 competition.


---
## ST3 — Image-Only Cross-Cultural Sentiment Prediction

Given only a painting and a language, predict that language's share of
positive annotators. Ranked by **within-painting cultural contrast**: for each
painting, the correlation between your predictions and the true shares across
its languages, averaged over paintings.

In [9]:
def st3_score(frame, predictions):
    """Pearson correlation across each painting's languages, averaged over paintings."""
    frame = frame.assign(q=predictions)
    scores = []
    for _, g in frame.groupby("painting"):
        if len(g) < 3 or g["p"].nunique() == 1:
            continue
        scores.append(0.0 if g["q"].nunique() == 1 else np.corrcoef(g["q"], g["p"])[0, 1])
    return np.mean(scores)

train_shares, val_shares = shares(train), shares(val)

Baseline: each language's average share on the training paintings. It ignores
the image, so your first improvement is to use it.

In [10]:
language_prior = train_shares.groupby("language")["p"].mean()
q = val_shares["language"].map(language_prior).fillna(0.7)
print("val contrast:", round(st3_score(val_shares, q), 3),
      "| log loss:", round(log_loss(q, val_shares["k"], val_shares["n"]), 3))

val contrast: 0.365 | log loss: 0.876


**Submission.** The input file has columns `painting,language,image`.

In [11]:
test = pd.read_csv(REPO / "phase_inputs" / "dev" / "st3_test.csv")
test["p_positive"] = test["language"].map(language_prior).fillna(0.7)
folder = Path("submission_st3")
folder.mkdir(exist_ok=True)
test[["painting", "language", "p_positive"]].to_csv(folder / "predictions.csv", index=False)
shutil.make_archive("submission_st3", "zip", folder)
print("Upload submission_st3.zip to the ST3 competition.")

Upload submission_st3.zip to the ST3 competition.


---
## Next steps

- The validation numbers above are approximate. Codabench's Development phase
  scores the same `val` paintings with the official scorer, so expect similar
  but not identical numbers there.
- For the Evaluation phase, change `dev` to `test` in the three submission
  cells once the test input files appear in `phase_inputs/test/`.
- Questions: [open an issue](https://github.com/artemotion-28/artemotion-28/issues).